# OpenRouter: Setup, Costs and a Simple Agent

Use OpenRouter with the OpenAI SDK, inspect a request's cost with the native SDK, then build an agent that calls a Python tool, with a short optional section on images, video and embeddings.

Run in order in Jupyter or [Google Colab](https://colab.research.google.com/github/BrightPool/udemy-prompt-engineering-course/blob/main/exploring_other_providers/openrouter_features_and_agent.ipynb), using Python 3.10 or later. The examples make paid API requests.

## 1. Set up OpenRouter and switch the base URL

1. Create an [OpenRouter account and API key](https://openrouter.ai/settings/keys).
2. Add [credits](https://openrouter.ai/settings/credits) to run the examples.
3. Enter your key in the hidden prompt below. An OpenRouter key is separate from an OpenAI key.

Keep the familiar OpenAI SDK. Set `base_url` to `https://openrouter.ai/api/v1`, use your OpenRouter key, and choose an OpenRouter model ID such as `openai/gpt-6.1-sol`. [Quickstart](https://openrouter.ai/docs/quickstart)

In [ ]:
# Pylint treats notebook cells as a script and pip magics as pass statements.
# pylint: disable=missing-module-docstring,wrong-import-position,wrong-import-order,await-outside-async,unnecessary-pass
%pip install openai --upgrade --quiet

In [ ]:
import os
from getpass import getpass
from openai import OpenAI

if not os.environ.get("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass("Enter your OpenRouter API key: ")

MODEL = "openai/gpt-6.1-sol"

In [ ]:
client = OpenAI(
    api_key=os.environ["OPENROUTER_API_KEY"],
    base_url="https://openrouter.ai/api/v1",
)

completion = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "Explain prompt engineering in one sentence."}],
    max_completion_tokens=2048,
    reasoning_effort="low",
)
print(completion.choices[0].message.content)

## 2. Use the OpenRouter SDK and see the cost

Now make a request with the [OpenRouter Python SDK](https://github.com/OpenRouterTeam/python-sdk). It uses the same key and model, with `chat.send()` instead of `chat.completions.create()`.

We pin compatible client and agent SDK versions for this lesson. OpenRouter returns token usage and the cost charged for this request in `usage`. [Usage accounting](https://openrouter.ai/docs/cookbook/administration/usage-accounting)

In [ ]:
%pip install openrouter==1.1.22 --upgrade --quiet

In [ ]:
from openrouter import OpenRouter

router = OpenRouter(api_key=os.environ["OPENROUTER_API_KEY"])
completion = router.chat.send(
    model=MODEL,
    messages=[{"role": "user", "content": "Explain prompt engineering in one sentence."}],
    max_completion_tokens=2048,
    reasoning_effort="low",
    stream=False,
)
print(completion.choices[0].message.content)

In [ ]:
usage = completion.usage
if usage is None:
    print("Usage and cost were not returned for this request.")
else:
    print(f"Input tokens: {usage.prompt_tokens}")
    print(f"Output tokens: {usage.completion_tokens}")
    if usage.cost is None:
        print("Cost was not returned for this request.")
    else:
        print(f"Request cost: ${usage.cost:.8f} USD")

## 3. Create a simple agent with a tool

Use the [OpenRouter Python agent SDK](https://github.com/OpenRouterTeam/python-agent) to give the model an `add_numbers` tool. The input schema describes its arguments; the Python function performs the calculation.

The SDK handles the loop: the model requests the tool, Python runs it, and the model receives the result and answers. The tool prints when it runs, so you can see it being used.

In [ ]:
%pip install openrouter-agent-sdk==0.8.0 openrouter==1.1.22 --upgrade --quiet

In [ ]:
from pydantic import BaseModel
from openrouter_agent import call_model, tool, step_count_is

class AddNumbersInput(BaseModel):
    """The two integers the model supplies to the tool."""

    a: int
    b: int


def add_numbers(params: AddNumbersInput, _context):
    """Calculate a sum and show when the agent runs this tool."""
    total = params.a + params.b
    print(f"Tool called: add_numbers({params.a}, {params.b}) -> {total}")
    return {"total": total}


addition_tool = tool(
    name="add_numbers",
    description="Add two integers and return their total.",
    input_schema=AddNumbersInput,
    execute=add_numbers,
)

In [ ]:
result = call_model(
    router,
    {
        "model": MODEL,
        "instructions": "Use add_numbers for calculations, then answer briefly with its result.",
        "input": "What is 12345 + 67890?",
        "tools": [addition_tool],
        "stop_when": step_count_is(3),
        "max_output_tokens": 2048,
        "reasoning": {"effort": "low"},
    },
)
print(await result.get_text())

## 4. Optional: images, video and embeddings

The same OpenRouter key can access specialist models through the gateway. Set the switch for each example to `True` to try it. Each enabled example makes a paid request; check the linked model pages for current pricing.

Images and video use dedicated REST endpoints here; embeddings use the native SDK. Each task needs a suitable model, rather than the text model from the earlier examples.

In [ ]:
%pip install requests --upgrade --quiet

RUN_IMAGE = False
RUN_VIDEO = False
RUN_EMBEDDINGS = False

In [ ]:
import base64
import requests
from IPython.display import Image, Video, display


def gateway_request(method, path, **kwargs):
    """Call a dedicated OpenRouter endpoint with the same API key."""
    response = requests.request(
        method,
        "https://openrouter.ai/api/v1" + path,
        headers={"Authorization": f"Bearer {os.environ['OPENROUTER_API_KEY']}"},
        timeout=180,
        **kwargs,
    )
    response.raise_for_status()
    return response

### Image generation

Send a prompt to the [Image API](https://openrouter.ai/docs/guides/overview/multimodal/image-generation), then decode and display the returned image. Model: [Seedream 4.5](https://openrouter.ai/bytedance-seed/seedream-4.5).

In [ ]:
if RUN_IMAGE:
    image_result = gateway_request("POST", "/images", json={
        "model": "bytedance-seed/seedream-4.5",
        "prompt": "A blue ceramic mug on a white studio background",
    }).json()
    display(Image(data=base64.b64decode(image_result["data"][0]["b64_json"])))

### Video generation

The [Video API](https://openrouter.ai/docs/guides/overview/multimodal/video-generation) returns a job ID because generation takes time. Submit once, then rerun the **status cell** below until the video is ready. Model: [Wan 2.7](https://openrouter.ai/alibaba/wan-2.7).

In [ ]:
if RUN_VIDEO:
    video_job = gateway_request("POST", "/videos", json={
        "model": "alibaba/wan-2.7",
        "prompt": "A blue ceramic mug slowly rotates on a white studio table.",
        "duration": 3,
        "resolution": "720p",
    }).json()
    print("Job ID:", video_job["id"])

In [ ]:
# Rerun this cell to check the existing job, without submitting another one.
if RUN_VIDEO:
    video_status = gateway_request("GET", f"/videos/{video_job['id']}").json()
    print("Status:", video_status["status"])
    if video_status["status"] == "completed":
        video_data = gateway_request("GET", f"/videos/{video_job['id']}/content").content
        display(Video(data=video_data, embed=True))
    elif video_status["status"] in {"failed", "cancelled", "expired"}:
        raise RuntimeError(video_status.get("error", "Video generation did not complete."))
    else:
        print("Still generating. Wait about 30 seconds, then rerun this status cell.")

### Embeddings

[Embeddings](https://openrouter.ai/docs/api/api-reference/embeddings/create-embeddings) turn text into numerical vectors for search and similarity. Model: [text-embedding-3-small](https://openrouter.ai/openai/text-embedding-3-small).

In [ ]:
if RUN_EMBEDDINGS:
    embedding_result = router.embeddings.generate(
        model="openai/text-embedding-3-small",
        input="Prompt engineering helps us give a model clearer instructions.",
        encoding_format="float",
    )
    vector = embedding_result.data[0].embedding
    print("Vector dimensions:", len(vector))
    print("First five values:", vector[:5])

## Summary

- Reuse the OpenAI SDK by changing the API key, base URL and model ID.
- Use the OpenRouter SDK to make a request and inspect its returned cost.
- Give an agent a Python tool; the agent SDK runs the tool and sends its result back to the model.
- Use specialist models through the same gateway for images, video and embeddings.